# 01 - Analisis Exploratorio de Datos (EDA)

Customer Churn Analysis - Curso ML2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

In [ ]:
df = pd.read_csv('../data/raw/telco_customer_churn.csv')
print(f'Shape: {df.shape}')
print(f'Columnas: {df.columns.tolist()}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Valores faltantes
missing = df.isnull().sum()
missing = missing[missing > 0]
print('Valores faltantes:')
print(missing if len(missing) > 0 else 'Ninguno')

In [ ]:
# TotalCharges puede tener espacios
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'Nulos en TotalCharges: {df["TotalCharges"].isnull().sum()}')

In [ ]:
# Distribucion de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x='Churn', ax=axes[0], palette='viridis')
axes[0].set_title('Distribucion de Churn')

df['Churn'].value_counts().plot.pie(autopct='%1.1f%%', ax=axes[1], colors=['#2ecc71', '#e74c3c'])
axes[1].set_title('Churn Rate')
axes[1].set_ylabel('')
plt.tight_layout()
plt.savefig('../reports/figures/churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Churn Rate: {df["Churn"].value_counts(normalize=True)["Yes"]:.1%}')

In [ ]:
# Distribucion de tenure por Churn
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(data=df, x='tenure', hue='Churn', kde=True, ax=axes[0, 0], palette='viridis')
axes[0, 0].set_title('Tenure por Churn')

sns.boxplot(data=df, x='Churn', y='MonthlyCharges', ax=axes[0, 1], palette='viridis')
axes[0, 1].set_title('Cargo Mensual por Churn')

sns.countplot(data=df, x='Contract', hue='Churn', ax=axes[1, 0], palette='viridis')
axes[1, 0].set_title('Contrato por Churn')
axes[1, 0].tick_params(axis='x', rotation=45)

sns.countplot(data=df, x='InternetService', hue='Churn', ax=axes[1, 1], palette='viridis')
axes[1, 1].set_title('Servicio Internet por Churn')

plt.tight_layout()
plt.savefig('../reports/figures/feature_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Variables categoricas vs Churn
cat_cols = df.select_dtypes(include='object').columns.drop(['customerID', 'Churn'])

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    if i < len(axes):
        churn_by_col = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean())
        churn_by_col.plot(kind='bar', ax=axes[i], color='coral')
        axes[i].set_title(f'Churn Rate by {col}')
        axes[i].set_ylabel('Churn Rate')
        axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../reports/figures/categorical_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlacion de variables numericas
numeric_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10, 8))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Matriz de Correlacion')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Resumen de hallazgos clave
print('=== HALLAZGOS CLAVE ===')
print(f'Total clientes: {len(df)}')
print(f'Churn rate: {df["Churn"].value_counts(normalize=True)["Yes"]:.1%}')
print(f'\nTop factores de churn:')
print(f'  - Contrato mes-a-mes: {df[df["Contract"]=="Month-to-month"]["Churn"].value_counts(normalize=True)["Yes"]:.1%} churn')
print(f'  - Sin tenure (< 12 meses): {df[df["tenure"]<12]["Churn"].value_counts(normalize=True)["Yes"]:.1%} churn')
print(f'  - Fiber optic: {df[df["InternetService"]=="Fiber optic"]["Churn"].value_counts(normalize=True)["Yes"]:.1%} churn')